# Optimization

In [1]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PowerTransformer
from scipy.optimize import fmin, minimize

In [3]:
# Load dataframes from binary files
base_dir = r'C:\FBr\Weiterbildung\Project\GitHub\sep24_cds_int_us-welding'
#base_dir = r'D:\Entwicklungen\share\DataScienceProject\sep24_cds_int_us-welding'

ft_param_path = os.path.join(base_dir, 'ft_files', '03_feat_parameters.ft')
df_params = pd.read_feather(ft_param_path)

dump_dir = os.path.join(base_dir, 'model_dumps')

In [4]:
# List of parameters
lst_param = df_params.columns.to_list()

lst_var = lst_param[1:4] + lst_param[5:7] + lst_param[8:24]
lst_nb_slots = [lst_param[4]] + [lst_param[7]]
lst_calc = lst_param[32:34]
lst_slot_class = lst_param[541:545]
lst_cols = lst_var + lst_calc + lst_slot_class

In [5]:
X = df_params[lst_cols]

In [6]:
# Get indices of Train-Test split
X_train, X_test, _, _ = train_test_split(X, X, test_size=0.2, random_state=55)
idx_train = X_train.index
idx_test = X_test.index

Target variables

In [7]:
## Long displacements
y_nrdd = df_params['long_disp_nrdd']
y_nrdd_train = y_nrdd.loc[idx_train]
y_nrdd_test = y_nrdd.loc[idx_test]

In [8]:
## All Frequencies
lst_freq = lst_param[86:122]
Y_freq = df_params[lst_freq]
Y_freq_test = Y_freq.loc[idx_test]

In [9]:
## Long frequencies
y_longfreq = df_params['freq_long']
y_longfreq_test = y_longfreq.loc[idx_test]

Long displacements

In [10]:
transf_longdisp = pickle.load(open(os.path.join(dump_dir, '5_model_neuralnetw_tuned_transformer.pkl'), 'rb'))
pipe_tuned_longdisp = pickle.load(open(os.path.join(dump_dir, '5_model_neuralnetw_tuned_long_displacements.pkl'), 'rb'))

# Apply the power transformer to the NRDD variable
w_test = transf_longdisp.transform(y_nrdd_test.values.reshape(-1, 1))

print ('Score of the test set:', pipe_tuned_longdisp.score(X_test, w_test))

Score of the test set: 0.8550440040374117


All frequencies

In [11]:
pipe_tuned_allfreq = pickle.load(open(os.path.join(dump_dir, '4_model_lgbm_tuned_all_frequencies.pkl'), 'rb'))
print ('Score of the test set:', pipe_tuned_allfreq.score(X_test, Y_freq_test))

Score of the test set: 0.9975845732050838


Long frequency

In [12]:
pipe_tuned_longfreq = pickle.load(open(os.path.join(dump_dir, '4_model_lgbm_tuned_long_frequencies.pkl'), 'rb'))
print ('Score of the test set:', pipe_tuned_longfreq.score(X_test, y_longfreq_test))

Score of the test set: 0.9933359528253154


## Select the design point with the minimum cost

In [13]:
# ID 1647 for example
id_01 = y_nrdd.argmin()

print (id_01)
print(y_nrdd[[id_01]])

x_01 = X.loc[id_01].tolist()

1647
1647    1.291901
Name: long_disp_nrdd, dtype: float64


In [14]:
pd.DataFrame([x_01], columns=lst_cols)

,dim_x,dim_y,dim_z,slot_x_length,slot_x_distance,slot_z_length,slot_z_distance,chamfer_x_dim_y,chamfer_x_dim_z,chamfer_y_dim_x,...,cut_x_depth,cut_z_start,cut_z_end,cut_z_depth,seg_x_out,seg_z_out,slot_2-2,slot_3-2,slot_4-1,slot_4-2
0,173.11,122.93,240.8,74.851,48.149,77.13,53.2,4.1767,14.899,6.5351,...,0.3487,23.429,63.553,2.1605,48.1765,59.955,0.0,0.0,0.0,1.0


In [15]:
# Predictions of all models
def fea_pred(x):
    """
    x is list
    """

    x1 = pd.DataFrame([x], columns=lst_cols)

    # Longitudinal frequency
    freq_long = pipe_tuned_longfreq.predict(x1)[0]
    # All frequencies
    all_freq = pipe_tuned_allfreq.predict(x1)[0]
    # NRDD

    nrdd_transf = pipe_tuned_longdisp.predict(x1)
    nrdd = transf_longdisp.inverse_transform(nrdd_transf)

    # Position of the long. mode
    delta_f = abs(all_freq-freq_long)
    id_long = delta_f.argmin()

    return freq_long, all_freq, nrdd[0][0], id_long


In [16]:
f_long, f_all, nrdd, idx = fea_pred(x_01) 
print (f_long, '\n', f_all, '\n', nrdd, '\n', idx)

19252.72063732115 
 [15260.27828227 15500.95683153 15599.54872674 15778.35841546
 15854.66728733 15908.47609677 15954.75439044 16023.66548765
 16033.27619731 16067.6331695  16142.872227   16403.63348786
 16443.07674304 16627.62247429 17097.06668295 17189.85312716
 17545.9414874  17679.7921517  17730.11106395 17902.47973162
 18414.82486541 18597.81419491 18662.80172641 18783.44023314
 18849.07959636 18873.26235529 19011.06742396 19091.63045942
 19279.80593494 19464.58127239 20375.83518321 20571.33844436
 20654.41997341 21226.59906278 22011.44300131 22064.09764223] 
 1.6892109 
 28


In [17]:
weights = {'f_long': 1/0.0006440096234678033, 'nrdd': 1/5.806682727065343, 'f_all': 1/0.12581713798880445}

In [ ]:
# cost function calculated with x
def f_cost(x, weights):
    weight_f_long = weights['f_long']
    weight_nrdd = weights['nrdd']
    weight_f_all = weights['f_all']

    f_long, f_all, nrdd, idx = fea_pred(x) 

    cost_f_long = weight_f_long * ((f_long-20e3)/20e3)**2
    cost_nrdd = weight_nrdd * nrdd

    f0 = f_all[idx]
    cost_f_below = (1/(f_all[idx-1]-f0))**2
    cost_f_above = (1/(f_all[idx+1]-f0))**2
    cost_f_all = weight_f_all * (cost_f_below + cost_f_above)

    return {'cost_f_long': cost_f_long, 'cost_nrdd': cost_nrdd, 'cost_f_all': cost_f_all}

In [19]:
f_cost(x_01, weights) 

{'cost_f_long': 2.1677721323431,
 'cost_nrdd': 0.2909080745621086,
 'cost_f_all': 0.0004572517349875102}

Calculate the cost functions for all design points. Then calculate the mean values in order to determine the optimal weights

In [45]:
def calc_cost_list(row):
    lst_row = row.to_list()
    return list(f_cost(lst_row, weights).values())

lst_row = df_params.loc[id, lst_cols].tolist()
calc_cost_list(df_params.loc[id, lst_cols])

[2.3252742980629297, 0.7908514826980171, 0.0013769790876557702]

In [ ]:
df_params[['cost_f_long', 'cost_nrdd', 'cost_f_all']] = df_params[lst_cols].apply(calc_cost_list, axis=1)

KeyboardInterrupt: 

In [ ]:
# for id in [0]: #df_params.index:
#     print ('Index: ', id, ' / ', len(df_params.index)-1)
#     lst_row = df_params.loc[id, lst_cols].tolist()
#     costs = f_cost(lst_row, weights)
    
#     df_params.loc[id, 'cost_f_long'] = costs['cost_f_long']
#     df_params.loc[id, 'cost_nrdd'] = costs['cost_nrdd']
#     df_params.loc[id, 'cost_f_all'] = costs['cost_f_all']

Index:  0  /  7989


In [20]:
print ('mean(cost_f_long):', df_params['cost_f_long'].mean())
print ('mean(cost_nrdd):', df_params['cost_nrdd'].mean())
print ('mean(cost_f_all):', df_params['cost_f_all'].mean())

KeyError: 'cost_f_long'

In [ ]:
#weights = {'f_long': 1/0.0006440096234678033, 'nrdd': 1/5.806682727065343, 'f_all': 1/0.12581713798880445}

In [ ]:
# Complete DataFrame with the cost values
df_params

,dp_no,dim_x,dim_y,dim_z,nb_slots_x,slot_x_length,slot_x_distance,nb_slots_z,slot_z_length,slot_z_distance,...,disp_long_399,sum_sensi,coupling_long,slot_2-2,slot_3-2,slot_4-1,slot_4-2,cost_f_long,cost_nrdd,cost_f_all
0,1001,175.06,121.95,240.31,4,75.993,43.235,2,78.219,59.508,...,0.151599,3.405414e-16,0.960466,0,0,0,1,2.325274,0.790851,0.001377
1,1002,173.45,122.77,238.13,4,81.168,45.859,2,78.924,57.208,...,0.143682,5.401730e-17,0.956635,0,0,0,1,NaN,NaN,NaN
2,1003,175.85,116.71,238.10,4,77.532,45.260,2,80.099,49.215,...,0.284010,2.678586e-18,0.986845,0,0,0,1,NaN,NaN,NaN
3,1004,173.51,116.37,238.30,4,82.428,51.007,2,74.757,50.840,...,0.186770,8.933099e-18,0.967459,0,0,0,1,NaN,NaN,NaN
4,1005,175.60,118.72,241.04,4,76.576,45.981,2,78.773,64.702,...,0.293833,8.025438e-18,0.991598,0,0,0,1,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7985,11995,175.18,119.50,239.53,2,78.314,80.925,2,77.038,54.535,...,0.175034,1.445371e-17,0.960601,1,0,0,0,NaN,NaN,NaN
7986,11996,176.27,117.29,240.59,2,77.933,71.773,2,79.789,63.279,...,0.332184,3.340311e-16,0.888124,1,0,0,0,NaN,NaN,NaN
7987,11997,174.01,117.81,239.26,2,80.402,81.905,2,79.398,60.146,...,0.266484,3.313740e-18,0.964230,1,0,0,0,NaN,NaN,NaN
7988,11998,174.61,119.50,239.41,2,74.848,74.107,2,74.870,57.402,...,-0.236469,1.190454e-15,0.606705,1,0,0,0,NaN,NaN,NaN


Find the design point x_0 with the minimal cost function

In [ ]:
# Note the slot class
df_params['cost'] = df_params['cost_f_long'] + df_params['cost_nrdd'] + df_params['cost_f_all']

In [24]:
plt.scatter(range(len(df_params)), np.log10(df_params['cost']))

KeyError: 'cost'

In [ ]:
idx_0 = df_params['cost'].idxmin()
print ('Index:', idx_0)

Index: 1152


In [49]:
idx_0 = 1152

In [50]:
df_params.loc[[idx_0]]

,dp_no,dim_x,dim_y,dim_z,nb_slots_x,slot_x_length,slot_x_distance,nb_slots_z,slot_z_length,slot_z_distance,...,disp_long_399,sum_sensi,coupling_long,slot_2-2,slot_3-2,slot_4-1,slot_4-2,cost_f_long,cost_nrdd,cost_f_all
1152,2153,175.51,119.21,241.21,4,75.641,53.089,2,76.463,49.445,...,0.290752,4.342749e-18,0.991657,0,0,0,1,NaN,NaN,NaN


In [108]:
xvar_0 = df_params.loc[idx_0, lst_var].tolist()
xvar_55 = df_params.loc[55, lst_var].tolist()

In [109]:
df_params.loc[idx_0, lst_cols]

dim_x               175.51
dim_y               119.21
dim_z               241.21
slot_x_length       75.641
slot_x_distance     53.089
slot_z_length       76.463
slot_z_distance     49.445
chamfer_x_dim_y     11.367
chamfer_x_dim_z     8.8571
chamfer_y_dim_x     10.448
chamfer_y_dim_z     6.0016
chamfer_z_dim_x     7.8842
chamfer_z_dim_y     13.908
elefoot_x          0.76625
elefoot_z          0.23867
cut_x_start         30.419
cut_x_end           70.146
cut_x_depth          4.489
cut_z_start         29.596
cut_z_end           63.282
cut_z_depth         1.3673
seg_x_out          40.9715
seg_z_out          63.0325
slot_2-2                 0
slot_3-2                 0
slot_4-1                 0
slot_4-2                 1
Name: 1152, dtype: object

In [156]:
# function x_var --> x
def f_xvar2x(xvar, nb_slots_x, nb_slots_z):

    xvar = np.array(xvar, dtype=float).copy()

    dim_z = xvar[2]
    dim_x = xvar[0]
    slot_x_distance = xvar[4]
    slot_z_distance = xvar[6]
    seg_x_out = (dim_z-(nb_slots_x-1)*slot_x_distance)/2
    seg_z_out = (dim_x-(nb_slots_z-1)*slot_z_distance)/2
    x_calc = [seg_x_out, seg_z_out]

    if (nb_slots_x == 2) & (nb_slots_z == 2):
        x_slot_classes = [1, 0, 0, 0]
    if (nb_slots_x == 3) & (nb_slots_z == 2):
        x_slot_classes = [0, 1, 0, 0]
    if (nb_slots_x == 4) & (nb_slots_z == 1):
        x_slot_classes = [0, 0, 1, 0]
    if (nb_slots_x == 4) & (nb_slots_z == 2):
        x_slot_classes = [0, 0, 0, 1]

    #return xvar + x_calc + x_slot_classes
    return np.concatenate([xvar, x_calc, x_slot_classes])

f_xvar2x(xvar_0, 4, 2)

array([1.75510e+02, 1.19210e+02, 2.41210e+02, 7.56410e+01, 5.30890e+01,
       7.64630e+01, 4.94450e+01, 1.13670e+01, 8.85710e+00, 1.04480e+01,
       6.00160e+00, 7.88420e+00, 1.39080e+01, 7.66250e-01, 2.38670e-01,
       3.04190e+01, 7.01460e+01, 4.48900e+00, 2.95960e+01, 6.32820e+01,
       1.36730e+00, 4.09715e+01, 6.30325e+01, 0.00000e+00, 0.00000e+00,
       0.00000e+00, 1.00000e+00])

In [159]:
# new cost function calculated with x_var
def f_cost2(xvar, weight_f_long, weight_nrdd, weight_f_all):

    x = f_xvar2x(xvar, 4, 2)

    f_long, f_all, nrdd, idx = fea_pred(x) 

    cost_f_long = weight_f_long * ((f_long-20e3)/20e3)**2
    cost_nrdd = weight_nrdd * nrdd

    f0 = f_all[idx]
    eps = 1e-8
    cost_f_below = (1/(f_all[idx-1]-f0) + eps)**2
    cost_f_above = (1/(f_all[idx+1]-f0) + eps)**2
    cost_f_all = weight_f_all * (cost_f_below + cost_f_above)
    cost_sum = cost_f_long + cost_nrdd + cost_f_all

    if not np.isfinite(cost_sum):
        print("Non-finite cost detected!", xvar)
        return 1e9

    print('xvar:', xvar)
    print('cost:', cost_sum)

    return cost_sum
    #return {'cost_f_long': cost_f_long, 'cost_nrdd': cost_nrdd, 'cost_f_all': cost_f_all}

# Test the new function
f_cost2(xvar_0, weights['f_long'], weights['nrdd'], weights['f_all'])

xvar: [175.51, 119.21, 241.21, 75.641, 53.089, 76.463, 49.445, 11.367, 8.8571, 10.448, 6.0016, 7.8842, 13.908, 0.76625, 0.23867, 30.419, 70.146, 4.489, 29.596, 63.282, 1.3673]
cost: 0.29596161675608323


0.29596161675608323

In [62]:
# Compare with the full point
x_0 = df_params.loc[idx_0, lst_cols].tolist()
f_cost(x_0, weights)

{'cost_f_long': 0.009187046878192533,
 'cost_nrdd': 0.2864489067096137,
 'cost_f_all': 0.0003256634491888921}

In [170]:
# Definitions of the bounds
var_bounds = [(173.0, 177.0), (115.0, 125.0), (238.0, 242.0), (74.0, 83.0), (41.0, 55.0), (74.0, 83.0), (49.0, 67.0), (0.1, 20.0), (0.1, 20.0), (0.1, 20.0), (0.1, 20.0),
          (0.1, 20.0), (0.1, 20.0), (0.1, 5.0), (0.1, 5.0), (20.0, 60.0), (60.0, 100.0), (0.1, 5.0), (20.0, 45.0), (45.0, 70.0), (0.1, 5.0)]

In [86]:
weight_f_long = 1552.7718275625957
weight_nrdd = 0.17221536753488045
weight_f_all = 7.948042818212752

In [ ]:
bres = minimize(
    f_cost2,
    xvar_0,
    args=(weight_f_long, weight_nrdd, weight_f_all),  # constants
    method='L-BFGS-B',   # supports bounds
    bounds=var_bounds,
    options={'maxiter': 500}
)
print('Success:', res.success)
print('Optimal x:', res.x)
print('Optimal cost:', res.fun)

xvar: [1.7551e+02 1.1921e+02 2.4121e+02 7.5641e+01 5.3089e+01 7.6463e+01
 4.9445e+01 1.1367e+01 8.8571e+00 1.0448e+01 6.0016e+00 7.8842e+00
 1.3908e+01 7.6625e-01 2.3867e-01 3.0419e+01 7.0146e+01 4.4890e+00
 2.9596e+01 6.3282e+01 1.3673e+00]
cost: 0.29596161675608323
xvar: [1.7551e+02 1.1921e+02 2.4121e+02 7.5641e+01 5.3089e+01 7.6463e+01
 4.9445e+01 1.1367e+01 8.8571e+00 1.0448e+01 6.0016e+00 7.8842e+00
 1.3908e+01 7.6625e-01 2.3867e-01 3.0419e+01 7.0146e+01 4.4890e+00
 2.9596e+01 6.3282e+01 1.3673e+00]
cost: 0.29596161675608323
xvar: [1.7551e+02 1.1921e+02 2.4121e+02 7.5641e+01 5.3089e+01 7.6463e+01
 4.9445e+01 1.1367e+01 8.8571e+00 1.0448e+01 6.0016e+00 7.8842e+00
 1.3908e+01 7.6625e-01 2.3867e-01 3.0419e+01 7.0146e+01 4.4890e+00
 2.9596e+01 6.3282e+01 1.3673e+00]
cost: 0.29596161675608323
xvar: [1.7551e+02 1.1921e+02 2.4121e+02 7.5641e+01 5.3089e+01 7.6463e+01
 4.9445e+01 1.1367e+01 8.8571e+00 1.0448e+01 6.0016e+00 7.8842e+00
 1.3908e+01 7.6625e-01 2.3867e-01 3.0419e+01 7.0146e+01 

In [169]:
res = differential_evolution(
    f_cost2,
    args=(weight_f_long, weight_nrdd, weight_f_all),  # constants
    bounds=var_bounds,
    polish=False
)
print('Success:', res.success)
print('Optimal x:', res.x)
print('Optimal cost:', res.fun)

xvar: [1.00175632e+02 1.16082685e+02 2.38870035e+02 7.97477140e+01
 5.24094573e+01 7.41238817e+01 5.15745078e+01 1.72530132e+01
 1.23136585e+01 1.00562291e+01 4.10288142e+00 1.43920254e+01
 1.43929030e+01 7.67464459e-01 1.15509771e-01 5.52377984e+01
 7.80739171e+01 2.29673080e+00 3.87994735e+01 5.86383290e+01
 8.09104882e-01]
cost: 2.219818794241497
xvar: [161.81676706 122.24310042 240.04280816  77.03393377  41.40831922
  80.12858215  66.92664192   4.0667817   10.33332673  15.74243718
   7.3283659   17.35054439  16.31598228   4.22008308   0.79919482
  41.2218046   77.52514586   0.27705223  36.4758737   55.97503089
   4.04976981]
cost: 2.453580579536631
xvar: [134.02849476 120.08519714 240.27964571  74.81802708  54.30567312
  77.66105908  66.86783269   1.16658135   7.20916349   4.82526203
  14.33825363   2.30990888  13.42023622   0.68036294   1.36512754
  30.44762298  85.9207987    0.70369854  20.73108243  62.96811514
   4.68083553]
cost: 1.5421350832707084
xvar: [112.26735669 122.40586

KeyboardInterrupt: 

In [171]:
for i in range(len(xvar_0)):
    x_test = xvar_0.copy()
    x_test[i] += 0.5  # small perturbation
    c1 = f_cost2(xvar_0, weight_f_long, weight_nrdd, weight_f_all)
    c2 = f_cost2(x_test, weight_f_long, weight_nrdd, weight_f_all)
    print(f"Var {i}: Δcost = {c2 - c1}")

xvar: [175.51, 119.21, 241.21, 75.641, 53.089, 76.463, 49.445, 11.367, 8.8571, 10.448, 6.0016, 7.8842, 13.908, 0.76625, 0.23867, 30.419, 70.146, 4.489, 29.596, 63.282, 1.3673]
cost: 0.29596161675608323
xvar: [176.01, 119.21, 241.21, 75.641, 53.089, 76.463, 49.445, 11.367, 8.8571, 10.448, 6.0016, 7.8842, 13.908, 0.76625, 0.23867, 30.419, 70.146, 4.489, 29.596, 63.282, 1.3673]
cost: 0.2986266693425681
Var 0: Δcost = 0.0026650525864848618
xvar: [175.51, 119.21, 241.21, 75.641, 53.089, 76.463, 49.445, 11.367, 8.8571, 10.448, 6.0016, 7.8842, 13.908, 0.76625, 0.23867, 30.419, 70.146, 4.489, 29.596, 63.282, 1.3673]
cost: 0.29596161675608323
xvar: [175.51, 119.71, 241.21, 75.641, 53.089, 76.463, 49.445, 11.367, 8.8571, 10.448, 6.0016, 7.8842, 13.908, 0.76625, 0.23867, 30.419, 70.146, 4.489, 29.596, 63.282, 1.3673]
cost: 0.28691580326936195
Var 1: Δcost = -0.009045813486721288
xvar: [175.51, 119.21, 241.21, 75.641, 53.089, 76.463, 49.445, 11.367, 8.8571, 10.448, 6.0016, 7.8842, 13.908, 0.76625,